# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neha-raniii/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/neha-raniii/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print(f"{len(df):,} rows loaded")

30,000 rows loaded


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — "The Freshness Multiplier" (365+ refresh cohort): The paper claims that among pages older than a year, the cohort refreshed in the last 30 days shows a 1.6x health lift and 52x impression lift versus pages last updated 181-360 days ago.

Methodology question: This 52x figure comes from a specific cohort comparison, but the paper's own caveat notes the broader 361+ freshness bucket has only 802 pages with just 21 declining - a very small and imbalanced sample. My question: was this 52x lift tested for statistical significance the same way the paper's other findings were (the paper reports p<0.001 for six other tests, but does not list this specific 365+ refresh comparison in that significance table)? A 52x multiplier from a small cohort is the kind of number that can be driven by a handful of outlier pages rather than a stable, general pattern.

Finding 2 — "Growth Prediction" model: The paper reports the growth model gets 90% accuracy on unseen pages from the same brands, but only 75% accuracy on completely unseen brands (range 64%-85%).

Methodology question: A 15-point accuracy drop between same-brand and new-brand evaluation is a meaningful signal that the model is partly learning brand-specific patterns, not a fully general one. My question: was the "same brand, new pages" evaluation using a grouped-by-page split within brand, or could pages from the same time window/content batch appear in both train and test? If pages published close together share unusual similarity (same campaign, same season), that could inflate the same-brand number beyond what a stricter time-aware split would show -

In [2]:
# These are methodology questions about the FlyRank paper, not computations on my own data.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before/after: re-running my Week-5 Random Forest model, comparing a plain random split (before - the naive approach) against the client-holdout grouped split I actually used (after - the honest approach), on the same features and same metric (Precision@50).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

feature_cols = ['impressions_90d', 'avg_position', 'ctr', 'word_count',
                 'days_since_last_update', 'search_volume', 'engagement_rate']
model_df = df.dropna(subset=feature_cols + ['client_id']).copy()

# BEFORE: naive random split (no grouping - pages from same client can appear in both sets)
X = model_df[feature_cols]
y = model_df['is_declining_label']
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.25, random_state=42)
model_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_r, y_tr_r)
p50_random = precision_at_k(model_random.predict_proba(X_te_r)[:, 1], y_te_r.values, 50)

# AFTER: honest client-holdout split (same as Week 5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_id']))
train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]
model_honest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(
    train_df[feature_cols], train_df['is_declining_label'])
p50_honest = precision_at_k(model_honest.predict_proba(test_df[feature_cols])[:, 1],
                              test_df['is_declining_label'].values, 50)

comparison = pd.DataFrame({
    'Split type': ['BEFORE: random (naive)', 'AFTER: client-holdout (honest)'],
    'Precision@50': [p50_random, p50_honest]
})
print(comparison.to_string(index=False))


Result: The naive random split showed Precision@50 = 0.98 - nearly perfect - while the honest client-holdout split showed 0.74. That 24-point gap is almost certainly because a random split let pages from the same client sit in both train and test, letting the model partly memorize client-specific patterns instead of learning a signal that generalizes to a new client. 0.74 (the honest number) is the one I trust and the one reported in my Week-5 work; 0.98 would have been a misleading headline number.

This directly echoes the methodology question I raised about the FlyRank paper's Growth Prediction model (Finding 2): the 90% same-brand vs 75% new-brand accuracy gap in that paper is the same shape of problem I just reproduced on my own data, which makes me more confident that question was a fair one to ask.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage audit on my final feature set (impressions_90d, avg_position, ctr, word_count, days_since_last_update, search_volume, engagement_rate): checking each one was knowable before the decision point, and confirming trend_direction/trend_pct (which the label is built from) were never included.

In [ ]:
print("Feature set used in the model:", feature_cols)
print("\nLabel-derived fields (must NOT be in feature set):", ['trend_direction', 'trend_pct'])
print("Overlap check:", set(feature_cols) & {'trend_direction', 'trend_pct'})
print("\nIf the overlap above is an empty set, no direct label leakage is present in the feature list.")


Leakage audit result: CLEAN. None of the seven features overlap with the label-derived fields (trend_direction, trend_pct). All seven features (impressions_90d, avg_position, ctr, word_count, days_since_last_update, search_volume, engagement_rate) are observed signals that would have been knowable at the decision point, not values computed from the outcome itself. This matches the leakage check I ran in Week 3 on the warehouse data, where I deliberately added ctr_mar as a leaky feature, watched AUC jump to a suspicious 1.0, and removed it - the same discipline applied here confirms this feature set is clean.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
"""My boldest original claim (from Week 5 / capstone paper): "The Random Forest clearly beats the Week-4 baseline rule" (stated plainly, without qualification).

Rewritten in safe language: On this dataset's client-holdout split, the Random Forest model showed a higher observed Precision@50 (0.74) than the Week-4 baseline rule (0.56) - a directional result on one split of the 30,000-row starter dataset. This is decision-support evidence that the model may generalize better than the hand-written rule, not proof that it will outperform on the full warehouse, on a different time period, or on a future-window label. The 0.98 vs 0.74 gap I found in this notebook is itself a reminder that even a "correct-looking" number needs its validation design checked before it's trusted."""


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.